
---

# 📘 **1 — Project Introduction**

# 🎧 AI-Powered Speaker Recognition System

This project detects **which speaker is talking** from a given audio clip using:

* 🎤 **Audio (.wav) upload**
* 🤖 **MFCC Feature Extraction**
* 🧠 **Deep Learning CNN Classifier**
* 👤 **Multi-Speaker Identification**
* 🌍 **Flask + ngrok Public Deployment**

### ✅ **Project Features**

* Automatic speaker label extraction
* MFCC generation (40-dimensional)
* CNN model training & validation
* Accuracy, loss visualization
* Flask Web UI for speaker prediction
* Public URL using ngrok

### 🎯 **Goal of this Project**

To build an end-to-end **Speaker Recognition System** where user uploads a `.wav` file and the system identifies:

✔ WHO spoke
✔ HOW CONFIDENT the model is

---

# 📘 **3 — Mount Google Drive & Load Dataset**

This step:

* Mounts Google Drive
* Reads the speaker dataset folder
* Displays number of speakers and samples

---

### ✅ **CELL 2: Mount Drive & Set Dataset Path**


---

# ✅ **Use Kaggle Dataset Instead of Google Drive (Highly Recommended!)**

Yes — you can **100% use the Kaggle dataset directly** instead of mounting Google Drive.
This is **cleaner, faster, more professional**, and works on any system without Drive.

### 🎧 Dataset Link (50-Speakers Audio Dataset):

👉 **[https://www.kaggle.com/datasets/vjcalling/speaker-recognition-audio-dataset](https://www.kaggle.com/datasets/vjcalling/speaker-recognition-audio-dataset)**

Using Kaggle removes the need to manually upload audio files or depend on Google Drive paths.

---

# ❌ **REMOVE This Old Code**

You will *completely remove* this block:

```python
#from google.colab import drive
#drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/Sasi Projects/50_speakers_audio_data/"
```

Why?
✔ Drive is slow
✔ Sometimes disconnects
✔ Difficult to share/reproduce
✔ Kaggle dataset is structured & clean

---

# ✅ NEW — Professional Kaggle Setup (FINAL VERSION)

Below is the **exact replacement**, ready to paste into your notebook.

---

## 📘 **CELL 1 — Install Kaggle API**

```python
# ===============================
# ✅ CELL: Install Kaggle API
# ===============================

#!pip install -q kaggle
#print("Kaggle API installed!")
```

---

## 📘 **CELL 2 — Upload kaggle.json (One-time Step)**

1. Go to 👉 [https://www.kaggle.com/settings](https://www.kaggle.com/settings)
2. Find **API** section
3. Click **Create New Token**
4. You will get **kaggle.json**
5. Upload using:

```python
#from google.colab import files
#files.upload()  # ⬅️ Upload kaggle.json here
```

---

## 📘 **CELL 3 — Configure Kaggle Credentials**

```python
#!mkdir -p ~/.kaggle
#!cp kaggle.json ~/.kaggle/
#!chmod 600 ~/.kaggle/kaggle.json
```

---

## 📘 **CELL 4 — Download the 50-Speaker Dataset**

```python
# ===============================
# ✅ Download Speaker Dataset
# ===============================

#!kaggle datasets download -d vjcalling/speaker-recognition-audio-dataset
```

---

## 📘 **CELL 5 — Extract Dataset**

```python
#!unzip -q speaker-recognition-audio-dataset.zip
```

---

## 📘 **CELL 6 — Set Dataset Path**

```python
#import os

#DATASET_PATH = "/content/speaker-recognition-audio-dataset"

#print("✅ Dataset Path:", DATASET_PATH)
#print("🎧 Available Speaker Folders:", os.listdir(DATASET_PATH)[:10])
```

---

# 🎉 **Your Final Dataset Path**

Use this **in your entire project**:

```python
#DATASET_PATH = "/content/speaker-recognition-audio-dataset"
```

---

# ⭐ FINAL COMPARISON

| Old (Google Drive)     | New (Kaggle)                  |
| ---------------------- | ----------------------------- |
| Slow file access       | ⚡ Super fast reading          |
| Manual upload issues   | ✔ Auto-download every time    |
| Not reproducible       | ✔ Fully reproducible anywhere |
| Folder corruption risk | ✔ Clean dataset structure     |

---

# 👍 Your Notebook Will Now Look Like This (Example)

📘 1 — Install Kaggle
📘 2 — Upload kaggle.json
📘 3 — Download Speaker Dataset
📘 4 — Extract
📘 5 — Set Path → `DATASET_PATH = "/content/speaker-recognition-audio-dataset"`
📘 6 — Continue with MFCC extraction & CNN training

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 🔧 Change this path to where your dataset is stored in Drive
DATASET_PATH = "/content/drive/MyDrive/Sasi Projects/50_speakers_audio_data/"



---

# 📘 **2 — Install Dependencies**

This step installs all required libraries:

* `librosa` for audio processing
* `soundfile` for reading audio
* `tensorflow / torch` for deep learning
* `pyngrok` for deployment

---

### ✅ **CELL 1: Install All Dependencies**



In [ ]:
!pip install librosa soundfile torch torchvision torchaudio numpy matplotlib flask pyngrok tqdm

In [ ]:
import os, random, librosa, torch, torch.nn as nn
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


---

# 📘 **4 — Extract File Paths & Speaker Labels**

This step:

* Recursively scans `.wav` files
* Extracts speaker names from folder structure
* Builds numeric labels for each speaker

---

### ✅ **CELL 3: Scan Audio Files & Build Labels**
---


In [ ]:
import os, librosa, numpy as np

def extract_features(file_path, max_pad_len=100):
    audio, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    pad_width = max(0, max_pad_len - mfcc.shape[1])
    mfcc = np.pad(mfcc, pad_width=((0,0),(0,pad_width)), mode='constant')
    return mfcc

# ✅ Correct recursive search
all_wav_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for f in files:
        if f.endswith('.wav'):
            all_wav_files.append(os.path.join(root, f))

# Extract speaker IDs from folder names
speakers = sorted(list(set([os.path.basename(os.path.dirname(f)) for f in all_wav_files])))
speaker_to_label = {spk: i for i, spk in enumerate(speakers)}

print("✅ Total Speakers:", len(speakers))
print("✅ Total Audio Files:", len(all_wav_files))
print("Example File:", all_wav_files[0])



---

# 📘 **5 — Clean Dataset & Keep Only Valid Speakers**

This step:

* Counts files per speaker
* Removes speakers with only 1 sample
* Splits dataset into training/testing

---

### ✅ **CELL 4: Filter Speakers & Split Dataset**

In [ ]:
from collections import Counter

# Count number of audio files per speaker
counts = Counter([os.path.basename(os.path.dirname(f)) for f in all_wav_files])
print("🔍 Speaker counts (sample):", list(counts.items())[:10])

# Keep only speakers with at least 2 samples
valid_files = [f for f in all_wav_files if counts[os.path.basename(os.path.dirname(f))] >= 2]

# Rebuild labels for valid files
labels = [speaker_to_label[os.path.basename(os.path.dirname(f))] for f in valid_files]

print(f"✅ Speakers kept: {len(set([os.path.basename(os.path.dirname(f)) for f in valid_files]))}")
print(f"✅ Total usable audio files: {len(valid_files)}")

# Split safely
train_files, test_files, y_train, y_test = train_test_split(
    valid_files, labels, test_size=0.2, random_state=42, stratify=labels
)



---

# 📘 **6 — MFCC Feature Extraction**

This step:

* Loads audio
* Extracts MFCC features
* Pads to fixed length for CNN

---
### ✅ **CELL 5: MFCC Extraction Function**
---

In [ ]:
import librosa
import numpy as np
from tqdm import tqdm

def extract_features(file_path, n_mfcc=40, max_len=174):
    try:
        audio, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

        # Padding / trimming to fixed length
        if mfcc.shape[1] < max_len:
            pad_width = max_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, pad_width=((0,0), (0,pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_len]

        return mfcc
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Extract MFCCs for training and testing
X_train = []
X_test = []

print("🎧 Extracting features for training data...")
for f in tqdm(train_files):
    feat = extract_features(f)
    if feat is not None:
        X_train.append(feat)

print("🎧 Extracting features for testing data...")
for f in tqdm(test_files):
    feat = extract_features(f)
    if feat is not None:
        X_test.append(feat)

# Convert to numpy arrays
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"✅ Feature shapes — X_train: {X_train.shape}, X_test: {X_test.shape}")



---

# 📘 **7 — Generate MFCCs for Training & Testing**

This step:

* Extracts MFCCs for all audio files
* Converts everything into numpy arrays

---

### ✅ **CELL 6: Extract All MFCC Features**

---



In [ ]:
print("Before Fix — min label:", np.min(y_train))
print("Before Fix — max label:", np.max(y_train))
print("Unique Labels Sample:", sorted(list(set(y_train)))[:10])


---
# 📘 **8 — Fix Label Indexing**

Deep learning requires labels starting from **0**.

---

### ✅ **CELL 7: Normalize Labels**
---



In [ ]:
# Shift labels to start at 0 instead of 1
y_train = np.array(y_train) - np.min(y_train)
y_test = np.array(y_test) - np.min(y_test)

num_classes = len(np.unique(y_train))

print("After Fix — min label:", np.min(y_train))
print("After Fix — max label:", np.max(y_train))
print("✅ num_classes =", num_classes)




---

# 📘 **9 — Prepare Data for CNN & One-Hot Encode**

Converts MFCC → CNN input shape.

---

### ✅ **CELL 8: Prepare Tensor Inputs**

In [ ]:
import numpy as np

# Rebuild speaker labels cleanly from valid files
unique_speakers = sorted(list(set([os.path.basename(os.path.dirname(f)) for f in valid_files])))
speaker_to_newlabel = {spk: i for i, spk in enumerate(unique_speakers)}

# Apply new continuous labels
y = np.array([speaker_to_newlabel[os.path.basename(os.path.dirname(f))] for f in valid_files])

# Verify
print("Total unique labels:", len(np.unique(y)))
print("Label range:", np.min(y), "to", np.max(y))




---

# 📘 **10 — Build CNN Model for Speaker Recognition**

Uses:

* Conv2D
* BatchNorm
* MaxPooling
* Dense layers

---

### ✅ **CELL 9: Build CNN Model**

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(40, X_train_cnn.shape[2], 1)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',

In [ ]:
from sklearn.model_selection import train_test_split

train_files, test_files, y_train, y_test = train_test_split(
    valid_files, y, test_size=0.2, random_state=42, stratify=y
)

num_classes = len(np.unique(y_train))
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

print("✅ Shapes:", y_train_cat.shape, y_test_cat.shape)



---

# 📘 **11 — Train the CNN Model**

---

### ✅  **Train Model**


---

# 📘 **Evaluate Accuracy & Plot Graphs**

---

### ✅ **Evaluation & Visualization**

---



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
import matplotlib.pyplot as plt

# ---- Prepare data ----
X_train_cnn = np.array([extract_features(f) for f in train_files])
X_test_cnn = np.array([extract_features(f) for f in test_files])

# Reshape for CNN
X_train_cnn = X_train_cnn[..., np.newaxis]
X_test_cnn = X_test_cnn[..., np.newaxis]

print("✅ Data shapes:", X_train_cnn.shape, X_test_cnn.shape)

# ---- Build CNN model ----
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(40, X_train_cnn.shape[2], 1)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ---- Train model ----
history = model.fit(
    X_train_cnn, y_train_cat,
    validation_data=(X_test_cnn, y_test_cat),
    epochs=20,
    batch_size=16,
    verbose=1
)

# ---- Evaluate ----
test_loss, test_acc = model.evaluate(X_test_cnn, y_test_cat)
print(f"\n🎯 Test Accuracy: {test_acc*100:.2f}%")

# ---- Plot Accuracy & Loss ----
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.show()




---

# 📘 **14 — Flask Deployment Setup**

Creates:

* Flask app
* Prediction API
* HTML interface

---

### ✅ **CELL 13: Install & Import flask + ngrok**




---

## 📘  Run Flask Server & ngrok Deployment

This step:

* Stops existing Flask/ngrok
* Starts Flask Server
* Creates Public ngrok URL

# 🔐 ngrok Token Removed for Security

User should insert their own token.

# ===============================

# ✅  Run Server & ngrok

# ===============================

---

## 📘  Authenticate ngrok

This step:

* Authenticates ngrok with your account
* Enables secure public HTTPS access
* Prepares the system for live deployment

# ===============================

---

## 🌐 Ngrok Setup (Public Deployment)

Ngrok provides a **secure public HTTPS link** to your locally running Flask application.

🔐 **For security reasons, your ngrok token should NOT be shared publicly.**

### ✅ To Use Ngrok, Follow These Steps:

### 📌 Step 1 — Get Your Auth Token

Go to this link and copy your personal token:
👉 **[https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)**

---

### 📌 Step 2 — Add Token Inside Notebook

Paste your token in the following line:

```python
#from pyngrok import ngrok, conf

#conf.get_default().auth_token = "YOUR_NGROK_TOKEN_HERE"
```

---

### 📌 Step 3 — Start Ngrok Tunnel

```python
#public_url = ngrok.connect(8000)
#print("🌍 Public URL:", public_url)
```

✅ After running this, a **shareable public link** will appear here.
You can open it in your browser and access your Flask app from **anywhere in the world** 🌎

---

### ✅ Summary

✔ Secure HTTPS URL

✔ No port forwarding required

✔ Works on Google Colab

✔ Perfect for project demos, reviews, and viva

---


In [ ]:
!pip install flask pyngrok tensorflow librosa soundfile numpy tqdm

import os
import numpy as np
import tensorflow as tf
import librosa
from flask import Flask, request, render_template_string, jsonify
from pyngrok import ngrok

# =========================================================
# CONFIGURATION
# =========================================================
NGROK_AUTH_TOKEN = "30okk9QVCXc4jaeD6xQw8iqcsKd_inZ4Fa2tjFE1dhLvi9zm"  # ✅ your token
MODEL_PATH = "/content/speaker_recognition_cnn.h5"  # ✅ trained model path
DATASET_PATH = "/content/drive/MyDrive/Sasi Projects/50_speakers_audio_data"

# Speaker name mapping
speakers = sorted(os.listdir(DATASET_PATH))
id_to_speaker = {i: spk for i, spk in enumerate(speakers)}

# =========================================================
# CONNECT NGROK
# =========================================================
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000).public_url
print("🌍 Public URL:", public_url)

# =========================================================
# LOAD MODEL
# =========================================================
model = tf.keras.models.load_model(MODEL_PATH)
print("✅ Model loaded successfully!")

# =========================================================
# FEATURE EXTRACTION (MATCH TRAINING EXACTLY)
# =========================================================
def extract_features(file_path, n_mfcc=40, max_len=174):
    """Extract MFCCs exactly like during training"""
    try:
        y, sr = librosa.load(file_path, sr=16000)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        if mfcc.shape[1] < max_len:
            pad_width = max_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, pad_width=((0,0),(0,pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_len]
        mfcc = mfcc[..., np.newaxis]  # shape: (40, 174, 1)
        return np.expand_dims(mfcc, axis=0)  # shape: (1, 40, 174, 1)
    except Exception as e:
        print("❌ Error extracting features:", e)
        return None

# =========================================================
# FLASK APP
# =========================================================
app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
<title>🎧 Speaker Recognition</title>
<style>
body { font-family: Arial; background: #f7f9fc; text-align: center; margin-top: 100px; }
.container { background: white; padding: 30px; border-radius: 15px; box-shadow: 0 0 10px #ccc; width: 50%; margin: auto; }
h2 { color: #333; }
.result { margin-top: 20px; font-size: 20px; color: #007bff; }
</style>
</head>
<body>
<div class="container">
  <h2>🎧 Speaker Recognition System</h2>
  <form action="/predict" method="post" enctype="multipart/form-data">
    <input type="file" name="audio_file" accept=".wav" required><br><br>
    <input type="submit" value="Identify Speaker">
  </form>
  {% if speaker %}
    <div class="result">🧠 Predicted Speaker: <b>{{ speaker }}</b><br>
    🔹 Confidence: {{ conf }}%</div>
  {% endif %}
</div>
</body>
</html>
"""

@app.route("/", methods=["GET"])
def home():
    return render_template_string(HTML_TEMPLATE)

@app.route("/predict", methods=["POST"])
def predict():
    try:
        file = request.files.get("audio_file")
        if not file:
            return jsonify({"error": "No file uploaded"}), 400

        filepath = f"/tmp/{file.filename}"
        file.save(filepath)

        features = extract_features(filepath)
        if features is None:
            return jsonify({"error": "Failed to extract features"}), 500

        preds = model.predict(features)
        pred_class = int(np.argmax(preds))
        confidence = float(np.max(preds) * 100)

        speaker_name = id_to_speaker.get(pred_class, "Unknown Speaker")

        return render_template_string(HTML_TEMPLATE, speaker=speaker_name, conf=f"{confidence:.2f}")

    except Exception as e:
        print("🔥 Internal Error:", e)
        return jsonify({"error": str(e)}), 500

# =========================================================
# RUN APP
# =========================================================
app.run(port=5000)